# Preprocessing `EngSaf`

In [17]:
import os
from json import load
from sklearn.model_selection import train_test_split
import pandas as pd

### Read

In [18]:
def read(name):
    """
    Reads a csv file from the `raw` directory and returns it as a DataFrame.

    :param name: file name
    :return: pandas DataFrame
    """
    path    = "raw"
    file    = os.path.join(path, name)
    dataset = pd.read_csv(file)

    return dataset

### Clean

In [19]:
def clean(dataset, subset=None):
    """
    Cleans the dataset by removing NaNs, duplicates, and optionally deduplicating by a subset of columns.

    :param dataset: pandas DataFrame
    :param subset: column name
    :return: pandas DataFrame
    """
    if "Question_id" in dataset.columns: dataset.drop("Question_id", axis=1, inplace=True)

    dataset.dropna(inplace=True)
    dataset.drop_duplicates(inplace=True)

    if subset: dataset.drop_duplicates(subset=subset, inplace=True)

    return dataset

### Leaking & Unleaking

In [20]:
leakage_indices = load(open("leakage_indices.json"))

def leaking(dataset, name):
    """
    Getting subset of the dataset that corresponding to known leakage indices for a given name (e.g. unseen answers).

    :param dataset: pandas DataFrame
    :param name: dataset name
    :return: subset pandas DataFrame
    """
    index = leakage_indices[name]

    return dataset.iloc[index]


def unleaking(dataset, name):
    """
    Removes known leakage rows from the dataset for a given name and resets the index.

    :param dataset: pandas DataFrame
    :param name: dataset name
    :return: subset pandas DataFrame
    """
    index = dataset.index[leakage_indices[name]]

    dataset.drop(index=index, inplace=True)
    dataset.reset_index(drop=True, inplace=True)

    return dataset

### Concatenate

In [21]:
def concatenate(datasets):
    """
    Concatenates a list of datasets into a single DataFrame with reset index.

    :param datasets: pandas DataFrames
    :return: pandas DataFrame
    """
    dataset = pd.concat(datasets, ignore_index=True)

    return dataset

### Standardize

In [22]:
def standardize(dataset):
    """
    Standardizes column names, adds a mark scheme column, and resets the index.

    :param dataset: pandas DataFrame
    :return: pandas DataFrame
    """
    mark_scheme = {"0"             : "Incorrect response",
                   "1"             : "Partially correct response",
                   "2"             : "Correct response"}

    columns     = {"Question"      : "question",
                   "Correct Answer": "reference_answer",
                   "Student Answer": "student_answer",
                   "output_label"  : "score",
                   "feedback"      : "rationale"}

    dataset.rename(columns=columns, inplace=True)
    dataset.insert(loc=3, column="mark_scheme", value=str(mark_scheme))
    dataset.reset_index(drop=True, inplace=True)

    return dataset

### Split & Save

In [23]:
def split_save(dataset, phase):
    """
    Saves full training data and 25%, 50%, 75% stratified subsets;
    splits non-train data into 60% test and 40% validation, all saved as csv.

    :param dataset: pandas DataFrame
    :param phase: the name of the phase (e.g. train)
    :return: None
    """
    path = "clean"

    if phase == "train":
        phase_path   = os.path.join(path, phase)
        entries_path = os.path.join(phase_path, f"{dataset.shape[0]}_entries.csv")

        dataset.to_csv(entries_path, index=False)

        for ratio in [0.25, 0.5, 0.75]:
            _, entries = train_test_split(dataset, test_size=ratio, random_state=42, stratify=dataset["score"])

            entries.reset_index(drop=True, inplace=True)

            entries_path = os.path.join(phase_path, f"{entries.shape[0]}_entries.csv")

            entries.to_csv(entries_path, index=False)

        return None

    else:
        test, validation = train_test_split(dataset, test_size=0.4, random_state=42, stratify=dataset["score"])

        test      .reset_index(drop=True, inplace=True)
        validation.reset_index(drop=True, inplace=True)

        name_split = {"test": test, "validation": validation}

        for name, split in name_split.items():
            phase_path   = os.path.join(path, name)
            entries_path = os.path.join(phase_path, f"{split.shape[0]}_entries.csv")

            split.to_csv(entries_path, index=False)

        return None

## Pipline

#### 1. Read

In [24]:
raw_train       = read(name="train.csv")
unseen_answers  = read(name="unseen_answers.csv")
unseen_question = read(name="unseen_question.csv")
raw_val         = read(name="val.csv")

#### 2. Clean

In [25]:
raw_train       = clean(dataset=raw_train)
unseen_answers  = clean(dataset=unseen_answers)
unseen_question = clean(dataset=unseen_question)
raw_val         = clean(dataset=raw_val)

#### 3. Concatenate Leaking - Train

In [26]:
unseen_answers_leakages  = leaking(dataset=unseen_answers ,name="unseen_answers")
unseen_question_leakages = leaking(dataset=unseen_question,name="unseen_question")
raw_val_leakages         = leaking(dataset=raw_val        ,name="val")

datasets  = [raw_train, unseen_answers_leakages, unseen_question_leakages, raw_val_leakages]

raw_train = concatenate(datasets=datasets)

#### 4. Clean Subset - Train

In [27]:
raw_train = clean(dataset=raw_train, subset="Student Answer")

#### 5. Standardize - Train

In [28]:
train = standardize(dataset=raw_train)

#### 6. Concatenate Unleaking - Unseen

In [29]:
unseen_answers_non_leakages  = unleaking(dataset=unseen_answers ,name="unseen_answers")
unseen_question_non_leakages = unleaking(dataset=unseen_question,name="unseen_question")
raw_val_non_leakages         = unleaking(dataset=raw_val        ,name="val")

datasets = [unseen_answers_non_leakages, unseen_question_non_leakages, raw_val_non_leakages]

unseen   = concatenate(datasets=datasets)

#### 7. Standardize - Unseen

In [30]:
unseen = standardize(dataset=unseen)

#### 8. Split - Train

In [31]:
split_save(dataset=train , phase="train")

#### 9. Split - Unseen

In [32]:
split_save(dataset=unseen, phase="unseen")